# EXP-18: Classic ML Maximizado — XGBoost + CatBoost + ExtraTrees + Feature Engineering Agressivo

**Competition:** SPR 2026 — Mammography Report Classification  
**Task:** Classify mammography reports (Portuguese) into BI-RADS 0–6  
**Metric:** Macro F1  
**Runtime:** CPU only  

## Filosofia: extrair o maximo possivel de ML classico
1. **Novos modelos**: XGBoost e ExtraTrees (nunca testados no projeto)
2. **TF-IDF mais agressivo**: word 1-4 grams, char 2-7 grams, hashing trick
3. **Feature engineering expandido**: 40+ features clinicas, ratios, interacoes
4. **SMOTE** para oversampling de classes raras
5. **9 modelos diversos** no ensemble final
6. **Meta-learner** (LogReg) no topo do stacking
7. **Threshold tuning** agressivo + guardrails clinicos

In [ ]:
import os, re, hashlib, warnings, time, glob
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb

try:
    import xgboost as xgb
    HAS_XGB = True
    print('XGBoost available!')
except ImportError:
    !pip install -q xgboost
    import xgboost as xgb
    HAS_XGB = True
    print('XGBoost installed.')

try:
    from catboost import CatBoostClassifier
    HAS_CB = True
    print('CatBoost available!')
except ImportError:
    !pip install -q catboost
    from catboost import CatBoostClassifier
    HAS_CB = True
    print('CatBoost installed.')

try:
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
    print('imbalanced-learn available!')
except ImportError:
    !pip install -q imbalanced-learn
    from imblearn.over_sampling import SMOTE
    HAS_SMOTE = True
    print('imbalanced-learn installed.')

warnings.filterwarnings('ignore')
np.random.seed(42)

# --- Data Loading ---
def find_input(pattern):
    for root, dirs, files in os.walk('/kaggle/input'):
        for d in dirs:
            if pattern in d.lower():
                return os.path.join(root, d)
    hits = glob.glob(f'/kaggle/input/**/*{pattern}*', recursive=True)
    return hits[0] if hits else None

COMP_DIR = find_input('spr-2026-mammography')
assert COMP_DIR, f'Competition data not found! Contents: {os.listdir("/kaggle/input")}'

train = pd.read_csv(os.path.join(COMP_DIR, 'train.csv'))
test  = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))

N_FOLDS = 5
N_CLASSES = 7

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train["target"].value_counts().sort_index()}')

In [ ]:
# =============================================================================
# Cell 2: Text Preprocessing
# =============================================================================

def normalize_text(s):
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    return s

def extract_achados(s):
    match = re.search(r'achados[:\s]*(.*?)(?:impress[aã]o|conclus[aã]o|an[aá]lise comparativa|opini[aã]o|$)', s, flags=re.DOTALL)
    return match.group(1).strip() if match else ""

def extract_conclusion(s):
    match = re.search(r'(?:impress[aã]o|conclus[aã]o|opini[aã]o)[:\s]*(.*?)(?:an[aá]lise comparativa|recomenda|$)', s, flags=re.DOTALL)
    return match.group(1).strip() if match else ""

def clean_text(s):
    s = normalize_text(s)
    s = re.sub(r'\b(cm|mm)\b', ' \\1 ', s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

# Versao com abreviacoes expandidas (mais agressiva)
ABBREV_MAP = {
    'bi-rads': 'birads', 'calc.': 'calcificacao', 'nod.': 'nodulo',
    'dx.': 'diagnostico', 'lt.': 'lateral', 'cc': 'craniocaudal',
    'mlo': 'mediolateral', 'qse': 'quadrante_superoexterno',
    'qsi': 'quadrante_superointerno', 'qie': 'quadrante_inferoexterno',
    'qii': 'quadrante_inferointerno', 'rcc': 'retroareolar_craniocaudal',
}

def clean_text_aggressive(s):
    s = clean_text(s)
    for k, v in ABBREV_MAP.items():
        s = s.replace(k, v)
    return s

def stable_hash(s):
    return hashlib.md5(str(s).encode('utf-8')).hexdigest()

for df in [train, test]:
    df['normalized']  = df['report'].apply(normalize_text)
    df['achados']     = df['normalized'].apply(extract_achados)
    df['conclusion']  = df['normalized'].apply(extract_conclusion)
    df['clean_full']  = df['report'].apply(clean_text)
    df['clean_agg']   = df['report'].apply(clean_text_aggressive)
    df['clean_achados']    = df['achados'].apply(clean_text)
    df['clean_conclusion'] = df['conclusion'].apply(clean_text)

train['group'] = train['report'].apply(stable_hash)
y = train['target'].astype(int).values
groups = train['group'].values

print('Preprocessing done.')

In [ ]:
# =============================================================================
# Cell 3: Feature Engineering Expandido (40+ features)
# =============================================================================

def extract_birads_number(text):
    match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    return int(match.group(1)) if match else -1

def count_pattern(series, pattern):
    return series.str.count(pattern).fillna(0).astype(int)

def extract_dense_features(df):
    feat = pd.DataFrame(index=df.index)
    text = df['normalized']
    achados = df['achados']
    conclusion = df['conclusion']
    
    # === Structure features ===
    feat['report_length']     = text.str.len()
    feat['word_count']        = text.str.split().str.len().fillna(0).astype(int)
    feat['line_count']        = text.str.count('\n') + 1
    feat['achados_length']    = achados.str.len()
    feat['conclusion_length'] = conclusion.str.len()
    feat['num_findings']      = achados.str.count(r'[\n\-\*\u2022]').fillna(0).astype(int)
    feat['achados_ratio']     = feat['achados_length'] / (feat['report_length'] + 1)
    feat['conclusion_ratio']  = feat['conclusion_length'] / (feat['report_length'] + 1)
    feat['avg_word_length']   = text.apply(lambda x: np.mean([len(w) for w in x.split()]) if x.strip() else 0)
    feat['sentence_count']    = text.str.count(r'[.!?]') + 1
    feat['avg_sentence_len']  = feat['word_count'] / feat['sentence_count']
    
    # === Clinical keyword features (binary) ===
    feat['has_measurement']    = text.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    feat['has_spiculation']    = text.str.contains(r'espiculad', regex=True).astype(int)
    feat['has_distortion']     = text.str.contains(r'distor[cç][aã]o arquitetural', regex=True).astype(int)
    feat['has_biopsy']         = text.str.contains(r'bi[oó]psia|carcinoma|resultado de cine', regex=True).astype(int)
    feat['has_nodule']         = text.str.contains(r'n[oó]dulo', regex=True).astype(int)
    feat['has_calcification']  = text.str.contains(r'calcifica[cç]', regex=True).astype(int)
    feat['has_microcalc']      = text.str.contains(r'microcalcifica', regex=True).astype(int)
    feat['has_asymmetry']      = text.str.contains(r'assimetria', regex=True).astype(int)
    feat['has_lymphnode']      = text.str.contains(r'linfonodo|axilar', regex=True).astype(int)
    feat['has_suspicious']     = text.str.contains(r'suspeit[oa]|malign[oa]', regex=True).astype(int)
    feat['has_benign']         = text.str.contains(r'benign[oa]|cisto[s]?\b', regex=True).astype(int)
    feat['has_gross_calc']     = text.str.contains(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', regex=True).astype(int)
    feat['has_nodulo_espic']   = text.str.contains(r'n[oó]dulo\s+espiculad', regex=True).astype(int)
    feat['has_heterogeneo']    = text.str.contains(r'heterog[eê]ne', regex=True).astype(int)
    feat['has_irregular']      = text.str.contains(r'irregular', regex=True).astype(int)
    feat['has_bilateral']      = text.str.contains(r'bilateral', regex=True).astype(int)
    feat['has_mama_direita']   = text.str.contains(r'mama direita', regex=True).astype(int)
    feat['has_mama_esquerda']  = text.str.contains(r'mama esquerda', regex=True).astype(int)
    
    # === Clinical keyword COUNTS (nao apenas binario!) ===
    feat['n_nodulos']         = count_pattern(text, r'n[oó]dulo')
    feat['n_calcifications']  = count_pattern(text, r'calcifica[cç]')
    feat['n_measurements']    = count_pattern(text, r'\b(?:cm|mm)\b')
    feat['n_suspicious_words'] = count_pattern(text, r'suspeit|malign|irregular|espiculad|distor')
    feat['n_benign_words']     = count_pattern(text, r'benign|cisto|normal|habitual|preservad')
    
    # === BI-RADS explicit ===
    feat['has_birads_explicit'] = text.str.contains(r'bi-?rads|categoria\s*\d', regex=True).astype(int)
    feat['birads_mentioned']    = text.apply(extract_birads_number)
    
    # === Composite scores ===
    feat['risk_score'] = (
        feat['has_spiculation'] * 3 + feat['has_distortion'] * 3 +
        feat['has_nodulo_espic'] * 4 + feat['has_biopsy'] * 5 +
        feat['has_suspicious'] * 3 + feat['has_irregular'] * 2 +
        feat['has_heterogeneo'] * 1 + feat['has_microcalc'] * 2 +
        feat['has_asymmetry'] * 2 - feat['has_benign'] * 2 - feat['has_gross_calc'] * 2
    )
    feat['suspicious_benign_ratio'] = (feat['n_suspicious_words'] + 1) / (feat['n_benign_words'] + 1)
    
    # === Interacoes ===
    feat['nodule_x_irregular']  = feat['has_nodule'] * feat['has_irregular']
    feat['calc_x_suspicious']   = feat['has_calcification'] * feat['has_suspicious']
    feat['spic_x_distortion']   = feat['has_spiculation'] * feat['has_distortion']
    feat['measurement_x_nodule'] = feat['has_measurement'] * feat['has_nodule']
    
    return feat

dense_train = extract_dense_features(train)
dense_test  = extract_dense_features(test)
X_train_dense = csr_matrix(dense_train.values.astype(np.float32))
X_test_dense  = csr_matrix(dense_test.values.astype(np.float32))

print(f'Dense features: {X_train_dense.shape[1]}')

In [ ]:
# =============================================================================
# Cell 4: TF-IDF Agressivo (mais variantes de n-grams)
# =============================================================================

print('Building TF-IDF features (aggressive)...')

# --- Full text: word 1-4, char 2-7 (mais largo que antes) ---
tfidf_word = TfidfVectorizer(ngram_range=(1, 4), min_df=3, max_df=0.95, sublinear_tf=True, max_features=150000)
tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 7), min_df=3, max_df=0.95,
                              sublinear_tf=True, max_features=120000)

X_tr_word = tfidf_word.fit_transform(train['clean_full'])
X_te_word = tfidf_word.transform(test['clean_full'])
X_tr_char = tfidf_char.fit_transform(train['clean_full'])
X_te_char = tfidf_char.transform(test['clean_full'])
print(f'Full text: word={X_tr_word.shape[1]}, char={X_tr_char.shape[1]}')

# --- Achados: word 1-3, char 3-5 ---
tfidf_ach_word = TfidfVectorizer(ngram_range=(1, 3), min_df=2, max_df=0.95, sublinear_tf=True)
tfidf_ach_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_df=0.95,
                                  sublinear_tf=True, max_features=50000)
X_tr_ach_w = tfidf_ach_word.fit_transform(train['clean_achados'])
X_te_ach_w = tfidf_ach_word.transform(test['clean_achados'])
X_tr_ach_c = tfidf_ach_char.fit_transform(train['clean_achados'])
X_te_ach_c = tfidf_ach_char.transform(test['clean_achados'])
print(f'Achados: word={X_tr_ach_w.shape[1]}, char={X_tr_ach_c.shape[1]}')

# --- Conclusion: word 1-2 ---
tfidf_concl = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_tr_concl = tfidf_concl.fit_transform(train['clean_conclusion'])
X_te_concl = tfidf_concl.transform(test['clean_conclusion'])
print(f'Conclusion: word={X_tr_concl.shape[1]}')

# --- Aggressive version (abbreviation-expanded) ---
tfidf_agg = TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)
X_tr_agg = tfidf_agg.fit_transform(train['clean_agg'])
X_te_agg = tfidf_agg.transform(test['clean_agg'])
print(f'Aggressive: word={X_tr_agg.shape[1]}')

# Combinar
X_train_sparse_full = hstack([X_tr_word, X_tr_char, X_tr_ach_w, X_tr_ach_c, X_tr_concl, X_tr_agg]).tocsr()
X_test_sparse_full  = hstack([X_te_word, X_te_char, X_te_ach_w, X_te_ach_c, X_te_concl, X_te_agg]).tocsr()

# Sparse-only para SVC/Ridge (sem achados/conclusion separados - mais simples)
X_train_sparse_simple = hstack([X_tr_word, X_tr_char]).tocsr()
X_test_sparse_simple  = hstack([X_te_word, X_te_char]).tocsr()

# Full features (sparse + dense)
X_train_all = hstack([X_train_sparse_full, X_train_dense]).tocsr()
X_test_all  = hstack([X_test_sparse_full, X_test_dense]).tocsr()

# Dense-only para tree models (SVD + dense) — mais rapido
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=80, random_state=42)
X_tr_svd = svd.fit_transform(X_train_sparse_full)
X_te_svd = svd.transform(X_test_sparse_full)
X_tr_tree = np.hstack([X_tr_svd, dense_train.values.astype(np.float32)])
X_te_tree = np.hstack([X_te_svd, dense_test.values.astype(np.float32)])

print(f'\nTotal sparse features: {X_train_sparse_full.shape[1]}')
print(f'Tree features (SVD+dense): {X_tr_tree.shape[1]}')
print(f'All features: {X_train_all.shape[1]}')

In [ ]:
# =============================================================================
# Cell 5: Train 9 Diverse Base Models
# =============================================================================

gkf = GroupKFold(n_splits=N_FOLDS)
folds = list(gkf.split(X_train_sparse_full, y, groups))

oof_all = {}
te_all  = {}

def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def train_model(name, model_fn, X_tr, X_te, use_smote=False):
    """Train a model with GroupKFold CV, optionally with SMOTE."""
    print(f'\nTraining {name}...')
    oof = np.zeros((len(train), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test), N_CLASSES), dtype=np.float64)
    
    for fi, (tr_idx, va_idx) in enumerate(folds):
        X_fold_tr, y_fold_tr = X_tr[tr_idx], y[tr_idx]
        
        if use_smote and HAS_SMOTE:
            try:
                # SMOTE precisa de array denso se sparse
                if hasattr(X_fold_tr, 'toarray'):
                    # Muito grande para densificar? Usar SVD primeiro
                    pass  # Skip SMOTE for sparse — usar sampling_strategy
                else:
                    sm = SMOTE(random_state=42, k_neighbors=min(3, min(pd.Series(y_fold_tr).value_counts()) - 1))
                    X_fold_tr, y_fold_tr = sm.fit_resample(X_fold_tr, y_fold_tr)
            except Exception as e:
                pass  # Fallback to original data
        
        m = model_fn()
        m.fit(X_fold_tr, y_fold_tr)
        
        if hasattr(m, 'predict_proba'):
            oof[va_idx] = m.predict_proba(X_tr[va_idx])
            te_acc += m.predict_proba(X_te) / N_FOLDS
        else:
            oof[va_idx] = softmax(m.decision_function(X_tr[va_idx]))
            te_acc += softmax(m.decision_function(X_te)) / N_FOLDS
    
    f1 = f1_score(y, np.argmax(oof, axis=1), average='macro')
    print(f'  {name} OOF F1: {f1:.4f}')
    oof_all[name] = oof
    te_all[name] = te_acc
    return f1

t0 = time.time()

# 1. SVC C=1.0 (sparse full)
train_model('svc_1.0', lambda: CalibratedClassifierCV(
    LinearSVC(class_weight='balanced', C=1.0, random_state=42, max_iter=15000),
    cv=3, method='sigmoid'), X_train_sparse_full, X_test_sparse_full)

# 2. SVC C=0.5 (sparse simple — diversidade de features)
train_model('svc_0.5', lambda: CalibratedClassifierCV(
    LinearSVC(class_weight='balanced', C=0.5, random_state=123, max_iter=15000),
    cv=3, method='sigmoid'), X_train_sparse_simple, X_test_sparse_simple)

# 3. SVC C=2.0 (aggressive text)
X_tr_agg_sp = hstack([X_tr_agg, X_tr_char]).tocsr()
X_te_agg_sp = hstack([X_te_agg, X_te_char]).tocsr()
train_model('svc_agg', lambda: CalibratedClassifierCV(
    LinearSVC(class_weight='balanced', C=2.0, random_state=456, max_iter=15000),
    cv=3, method='sigmoid'), X_tr_agg_sp, X_te_agg_sp)

# 4. Ridge (sparse full)
train_model('ridge', lambda: RidgeClassifier(class_weight='balanced', alpha=1.0),
            X_train_sparse_full, X_test_sparse_full)

# 5. LogReg (all features)
train_model('logreg', lambda: LogisticRegression(
    class_weight='balanced', C=1.0, max_iter=5000,
    solver='lbfgs', multi_class='multinomial', random_state=42, n_jobs=-1),
    X_train_all, X_test_all)

# 6. LightGBM (tree features)
train_model('lgb', lambda: lgb.LGBMClassifier(
    class_weight='balanced', n_estimators=500, learning_rate=0.03,
    max_depth=7, num_leaves=63, subsample=0.8, colsample_bytree=0.6,
    random_state=42, n_jobs=-1, verbose=-1), X_tr_tree, X_te_tree)

# 7. XGBoost (tree features)
# Calcular sample weights para class balancing no XGBoost
class_counts = np.bincount(y)
xgb_class_weights = len(y) / (N_CLASSES * class_counts)
xgb_sample_weights = np.array([xgb_class_weights[c] for c in y])

print('\nTraining XGBoost...')
oof_xgb = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_xgb  = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = xgb.XGBClassifier(
        n_estimators=500, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss',
        verbosity=0
    )
    m.fit(X_tr_tree[tr_idx], y[tr_idx], sample_weight=xgb_sample_weights[tr_idx])
    oof_xgb[va_idx] = m.predict_proba(X_tr_tree[va_idx])
    te_xgb += m.predict_proba(X_te_tree) / N_FOLDS
xgb_f1 = f1_score(y, np.argmax(oof_xgb, axis=1), average='macro')
print(f'  XGBoost OOF F1: {xgb_f1:.4f}')
oof_all['xgb'] = oof_xgb; te_all['xgb'] = te_xgb

# 8. CatBoost (tree features + SMOTE)
print('Training CatBoost...')
oof_cb = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_cb  = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    X_cb_tr, y_cb_tr = X_tr_tree[tr_idx], y[tr_idx]
    if HAS_SMOTE:
        try:
            min_class = min(pd.Series(y_cb_tr).value_counts())
            if min_class >= 4:
                sm = SMOTE(random_state=42, k_neighbors=min(3, min_class - 1))
                X_cb_tr, y_cb_tr = sm.fit_resample(X_cb_tr, y_cb_tr)
        except:
            pass
    m = CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6, l2_leaf_reg=3,
        auto_class_weights='Balanced', random_seed=42, verbose=0, task_type='CPU'
    )
    m.fit(X_cb_tr, y_cb_tr, eval_set=(X_tr_tree[va_idx], y[va_idx]), early_stopping_rounds=50)
    oof_cb[va_idx] = m.predict_proba(X_tr_tree[va_idx])
    te_cb += m.predict_proba(X_te_tree) / N_FOLDS
cb_f1 = f1_score(y, np.argmax(oof_cb, axis=1), average='macro')
print(f'  CatBoost OOF F1: {cb_f1:.4f}')
oof_all['catboost'] = oof_cb; te_all['catboost'] = te_cb

# 9. ExtraTrees (tree features)
print('Training ExtraTrees...')
oof_et = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_et  = np.zeros((len(test), N_CLASSES), dtype=np.float64)
for fi, (tr_idx, va_idx) in enumerate(folds):
    m = ExtraTreesClassifier(
        n_estimators=500, max_depth=20, min_samples_leaf=5,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    m.fit(X_tr_tree[tr_idx], y[tr_idx])
    oof_et[va_idx] = m.predict_proba(X_tr_tree[va_idx])
    te_et += m.predict_proba(X_te_tree) / N_FOLDS
et_f1 = f1_score(y, np.argmax(oof_et, axis=1), average='macro')
print(f'  ExtraTrees OOF F1: {et_f1:.4f}')
oof_all['extratrees'] = oof_et; te_all['extratrees'] = te_et

elapsed = time.time() - t0
print(f'\nAll {len(oof_all)} models trained in {elapsed:.0f}s')

In [ ]:
# =============================================================================
# Cell 6: Two-Level Stacking — Meta-Learner
# =============================================================================

print('='*60)
print('TWO-LEVEL STACKING')
print('='*60)

model_names = list(oof_all.keys())
n_models = len(model_names)

# Stack OOF predictions: (n_train, n_models * 7) features para meta-learner
oof_stack = np.hstack([oof_all[name] for name in model_names])  # (n, 63)
te_stack  = np.hstack([te_all[name] for name in model_names])

print(f'Meta-features: {oof_stack.shape[1]} (from {n_models} models x {N_CLASSES} classes)')

# Meta-learner: LogisticRegression
oof_meta = np.zeros((len(train), N_CLASSES), dtype=np.float64)
te_meta  = np.zeros((len(test), N_CLASSES), dtype=np.float64)

for fi, (tr_idx, va_idx) in enumerate(folds):
    meta = LogisticRegression(class_weight='balanced', C=1.0, max_iter=3000,
                              solver='lbfgs', multi_class='multinomial', random_state=42)
    meta.fit(oof_stack[tr_idx], y[tr_idx])
    oof_meta[va_idx] = meta.predict_proba(oof_stack[va_idx])
    te_meta += meta.predict_proba(te_stack) / N_FOLDS
    fold_f1 = f1_score(y[va_idx], np.argmax(oof_meta[va_idx], axis=1), average='macro')
    print(f'  Meta fold {fi}: F1={fold_f1:.4f}')

meta_f1 = f1_score(y, np.argmax(oof_meta, axis=1), average='macro')
print(f'\nMeta-learner OOF F1: {meta_f1:.4f}')

# --- Also try Dirichlet weight search for comparison ---
print('\nDirichlet weight search...')
best_f1 = 0
best_weights = None
np.random.seed(42)
for _ in range(20000):
    raw = np.random.dirichlet(np.ones(n_models))
    w = np.round(raw / 0.05) * 0.05
    if w.sum() == 0: continue
    w = w / w.sum()
    blend = sum(w[i] * oof_all[model_names[i]] for i in range(n_models))
    score = f1_score(y, np.argmax(blend, axis=1), average='macro')
    if score > best_f1:
        best_f1 = score
        best_weights = w.copy()

print(f'Dirichlet best OOF F1: {best_f1:.4f}')
print(f'Best weights:')
for name, w in zip(model_names, best_weights):
    if w > 0:
        print(f'  {name}: {w:.2f}')

oof_weighted = sum(best_weights[i] * oof_all[model_names[i]] for i in range(n_models))
te_weighted  = sum(best_weights[i] * te_all[model_names[i]] for i in range(n_models))

# Hybrid: 50% meta + 50% weighted
oof_hybrid = 0.5 * oof_meta + 0.5 * oof_weighted
te_hybrid  = 0.5 * te_meta + 0.5 * te_weighted
hybrid_f1 = f1_score(y, np.argmax(oof_hybrid, axis=1), average='macro')
print(f'Hybrid (50% meta + 50% weighted) OOF F1: {hybrid_f1:.4f}')

# Pick best
results = {'meta': (meta_f1, oof_meta, te_meta),
           'weighted': (best_f1, oof_weighted, te_weighted),
           'hybrid': (hybrid_f1, oof_hybrid, te_hybrid)}
best_strategy = max(results, key=lambda k: results[k][0])
final_score, oof_blend, te_blend = results[best_strategy]
print(f'\nBest strategy: {best_strategy} (F1={final_score:.4f})')

In [ ]:
# =============================================================================
# Cell 7: Threshold Search + Clinical Guardrails + Submission
# =============================================================================

threshold_classes = [6, 5, 4, 3, 1, 0]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in threshold_classes:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for higher_cls in threshold_classes:
                if higher_cls == cls: break
                if higher_cls in thresholds:
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

baseline_preds = np.argmax(oof_blend, axis=1)
baseline_f1 = f1_score(y, baseline_preds, average='macro')
print(f'Baseline OOF macro-F1 (no thresholds): {baseline_f1:.4f}')

best_thresholds = {}
current_f1 = baseline_f1
for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_blend, trial)
        trial_f1 = f1_score(y, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f'Class {cls}: threshold={best_cls_thr:.2f}, F1={best_cls_f1:.4f}')
    else:
        print(f'Class {cls}: no improvement')

final_oof_preds = apply_thresholds(oof_blend, best_thresholds)
final_oof_f1 = f1_score(y, final_oof_preds, average='macro')
print(f'\nFinal OOF macro-F1 with thresholds: {final_oof_f1:.4f}')
print(f'Thresholds: {best_thresholds}')
print(classification_report(y, final_oof_preds, digits=4))

# --- Submission ---
test_preds = apply_thresholds(te_blend, best_thresholds)
test['target'] = test_preds

def apply_clinical_guardrails(row):
    text = str(row['report']).lower()
    pred = int(row['target'])
    birads_match = re.search(r'(?:bi-?rads|categoria)\s*:?\s*(\d)', text)
    if birads_match:
        mentioned = int(birads_match.group(1))
        if 0 <= mentioned <= 6: return mentioned
    if re.search(r'resultado de cine grau 3|carcinoma|neoplasia maligna', text): return 6
    if re.search(r'n[oó]dulo\s+espiculad', text) and pred < 4: return 5
    if 'espiculad' in text and 'distor' in text and pred < 4: return 5
    if re.search(r'calcifica[cç][aã]o grosseira|calcifica[cç][aã]o vascular', text):
        if pred >= 4 and not re.search(r'espiculad|suspeit|malign|distor', text): return 2
    return pred

test['target'] = test.apply(apply_clinical_guardrails, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('\nSubmission saved!')
print(f'Prediction distribution:\n{submission["target"].value_counts().sort_index()}')
print(submission)